[Reference](https://www.towardsdeeplearning.com/rf-detr-a-smaller-model-that-beats-the-biggest-yolo-433f271382b6$0)

In [1]:
from transformers import pipeline
import torch

pipeline = pipeline("object-detection", model="Roboflow/rf-detr-medium", device_map="auto")

pipeline("http://images.cocodataset.org/val2017/000000039769.jpg")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/5.77k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/135M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/442 [00:00<?, ?B/s]

[{'score': 0.9466784000396729,
  'label': 'cat',
  'box': {'xmin': 7, 'ymin': 55, 'xmax': 316, 'ymax': 473}},
 {'score': 0.9389146566390991,
  'label': 'cat',
  'box': {'xmin': 347, 'ymin': 23, 'xmax': 639, 'ymax': 374}},
 {'score': 0.9313514232635498,
  'label': 'remote',
  'box': {'xmin': 40, 'ymin': 73, 'xmax': 175, 'ymax': 118}},
 {'score': 0.8544366955757141,
  'label': 'remote',
  'box': {'xmin': 334, 'ymin': 76, 'xmax': 370, 'ymax': 188}},
 {'score': 0.5297503471374512,
  'label': 'couch',
  'box': {'xmin': 1, 'ymin': 0, 'xmax': 639, 'ymax': 475}},
 {'score': 0.5117772817611694,
  'label': 'bed',
  'box': {'xmin': 1, 'ymin': 0, 'xmax': 639, 'ymax': 475}}]

In [3]:
from transformers import AutoImageProcessor, AutoModelForObjectDetection
from PIL import Image
import supervision as sv

import httpx
from io import BytesIO
import torch

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(BytesIO(httpx.get(url).content))

image_processor = AutoImageProcessor.from_pretrained("Roboflow/rf-detr-medium")
model = AutoModelForObjectDetection.from_pretrained("Roboflow/rf-detr-medium", device_map="auto")

inputs = image_processor(images=image, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

results = image_processor.post_process_object_detection(
    outputs, target_sizes=torch.tensor([image.height, image.width]), threshold=0.3
)[0]

detections = sv.Detections.from_transformers(
    transformers_results=results, id2label=model.config.id2label
)

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

annotated_image = image.copy()
annotated_image = box_annotator.annotate(annotated_image, detections)
annotated_image = label_annotator.annotate(annotated_image, detections)

sv.plot_image(annotated_image)